[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/badaouihakimou/machine-learning-notebooks/blob/main/07_built_in_functions.ipynb)



# Les fonctions intégrées

Python fournit une septantaine de fonctions disponibles sans aucun import. On en
utilise déjà plusieurs depuis le début : `print`, `len`, `range`, `type`.

La liste complète est sur
[docs.python.org/3/library/functions.html](https://docs.python.org/3/library/functions.html).

## Le plan

| Section | Le sujet |
|---|---|
| 1 | Les fonctions numériques |
| 2 | `all`, `any` et la notion de valeur vraie |
| 3 | Convertir un type |
| 4 | Convertir une structure |
| 5 | `input` et le formatage de texte |
| 6 | Construire des noms de clés dynamiquement |
| 7 | Lire et écrire des fichiers |
| 8 | `map`, `filter`, `sorted` |

## Trois pièges annoncés

`round(2.5)` ne donne pas 3. `float(x)` ne convertit pas `x`. Et `all` sur une
liste de nombres ne dit pas ce qu'on croit.

Aucun des trois ne lève d'erreur.

Prérequis : les notebooks 02 à 06.

## 1. Les fonctions numériques

In [116]:
print('abs(-3) :', abs(-3))
print('round(3.14) :', round(3.14))
print('round(3.14159, 2) :', round(3.14159, 2))
print('pow(2, 10) :', pow(2, 10))

abs(-3) : 3
round(3.14) : 3
round(3.14159, 2) : 3.14
pow(2, 10) : 1024


In [117]:
liste = [0, 23, 14, -19]
print('max :', max(liste))
print('min :', min(liste))
print('sum :', sum(liste))
print('len :', len(liste))

max : 23
min : -19
sum : 18
len : 4


`max` et `min` acceptent aussi plusieurs arguments directs : `max(3, 7, 2)`.

### Le piège de round

`round` n'arrondit pas comme à l'école.

In [118]:
for x in [0.5, 1.5, 2.5, 3.5, 4.5]:
    print(f'round({x}) = {round(x)}')

round(0.5) = 0
round(1.5) = 2
round(2.5) = 2
round(3.5) = 4
round(4.5) = 4


`round(2.5)` donne 2, pas 3. Et `round(3.5)` donne bien 4.

Ce n'est pas un bug : c'est l'arrondi au pair le plus proche, la norme
IEEE 754 utilisée par tous les langages modernes. Quand la valeur est
exactement au milieu, on arrondit vers le nombre pair.

La raison est statistique : arrondir systématiquement vers le haut introduit un
biais qui s'accumule sur de grands volumes de données. En alternant, les erreurs
se compensent.

Un second piège, lié aux flottants du notebook 02 :

In [119]:
print('round(2.675, 2) =', round(2.675, 2))
print('2.675 vaut en réalité :', f'{2.675:.20f}')

round(2.675, 2) = 2.67
2.675 vaut en réalité : 2.67499999999999982236


On attend 2.68, on obtient 2.67. Parce que 2.675 n'est pas représentable
exactement en binaire sa valeur réelle est légèrement inférieure.

Pour un arrondi commercial strict, il faut le module `decimal` :

```python
from decimal import Decimal, ROUND_HALF_UP
Decimal('2.675').quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)
```

En analyse de données, ces écarts sont sans importance. En comptabilité, ils
comptent.

In [120]:
from decimal import Decimal, ROUND_HALF_UP
Decimal('2.675').quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)

Decimal('2.68')

## 2. all, any, et ce qui compte comme vrai

`all` renvoie `True` si tous les éléments sont vrais. `any`, si au moins un
l'est.

In [121]:
booleens = [True, True, False]

print('all :', all(booleens))
print('any :', any(booleens))

all : False
any : True


In [122]:
booleens = [False, False, False]

print('all :', all(booleens))
print('any :', any(booleens))

all : False
any : False


Sur des booléens, c'est limpide. Sur des nombres, le résultat surprend.

In [123]:
liste = [0, 23, 14, -19]

print('all :', all(liste))
print('any :', any(liste))

all : False
any : True


In [124]:
liste = [0, 0, 0, 0]

print('all :', all(liste))
print('any :', any(liste))

all : False
any : False


`all` renvoie `False` alors que la liste ne contient aucun `False`.

L'explication : Python considère certaines valeurs comme fausses même si ce ne
sont pas des booléens.

In [125]:
for valeur in [0, 1, -19, 0.0, '', 'texte', [], [0], {}, None]:
    print(f'{str(valeur):<8} -> {bool(valeur)}')

0        -> False
1        -> True
-19      -> True
0.0      -> False
         -> False
texte    -> True
[]       -> False
[0]      -> True
{}       -> False
None     -> False


Sont faux : `0`, `0.0`, `''`, `[]`, `{}`, `()`, `set()`, `None`, `False`.
Tout le reste est vrai, y compris `-19` et `[0]`.

C'est ce qu'on appelle la valeur de vérité, ou *truthiness*. On l'a déjà
rencontrée sans le nommer :

```python
if liste and liste[0] > 5: # notebook 03 : une liste vide est fausse
print(sortie or i) # notebook 03 : une chaîne vide est fausse
```

Dans `all(liste)`, c'est donc le zéro qui rend le résultat faux. Si tu voulais
tester le signe, il faut l'écrire :

```python
all(n > 0 for n in liste)
```

Un piège lié : `all([])` renvoie `True`. Sur une liste vide, « tous les
éléments sont vrais » est vrai par défaut. Ça peut masquer un problème si la
liste ne devrait pas être vide.

In [126]:
print('all([]) :', all([]))
print('any([]) :', any([]))

all([]) : True
any([]) : False


Et une conséquence utile : `True` vaut 1 et `False` vaut 0, donc `sum` sur des
booléens les compte.

In [127]:
notes = [12, 8, 15, 6, 18]
print('Nombre de notes >= 10 :', sum(n >= 10 for n in notes))

Nombre de notes >= 10 : 3


C'est un idiome fréquent, et c'est exactement pourquoi `data['survived'].mean()`
donnait le taux de survie dans le notebook Pandas.

## 3. Convertir un type

In [128]:
x = 10
print(type(x))

x = str(x)
print(x, type(x))

y = int('20')
print(y, type(y))

<class 'int'>
10 <class 'str'>
20 <class 'int'>


### Le piège : la conversion ne modifie rien

Voici l'erreur la plus fréquente de cette section.

In [129]:
x = 10
print('avant :', type(x))

float(x) # le résultat n'est stocké nulle part

print('après :', type(x), '<- inchangé')

avant : <class 'int'>
après : <class 'int'> <- inchangé


Les deux affichent `int`.

`float(x)` calcule un nouveau flottant, puis le jette faute de destination.
Un entier étant immuable, il ne peut pas être transformé sur place.

Il faut réassigner :

In [130]:
x = 10
x = float(x)
print(type(x))

<class 'float'>


C'est le même principe que `sorted` contre `sort` au notebook 04 : une fonction
qui renvoie un résultat exige une réassignation, une méthode qui modifie sur
place non.

### Toutes les conversions n'aboutissent pas

In [131]:
try:
    int('Bonjour')
except ValueError as e:
    print('ValueError :', e)

print(int('42')) # celle-ci fonctionne
print(int(3.99)) # tronque, n'arrondit pas
print(int('1010', 2)) # base 2 : 1010 en binaire vaut 10

ValueError : invalid literal for int() with base 10: 'Bonjour'
42
3
10


Le second argument de `int` précise la base de lecture :

```python
int('1010', 2) # 10
int('ff', 16) # 255
```

Pour convertir un texte saisi par un utilisateur, il faut prévoir l'échec :

```python
try:
    valeur = int(saisie)
except ValueError:
    print('Ce n\'est pas un nombre')
```

### Les bases numériques

In [132]:
print('bin(10) :', bin(10))
print('oct(10) :', oct(10))
print('hex(10) :', hex(10))
print('retour  :', int('0b1010', 2), int('0xa', 16))

bin(10) : 0b1010
oct(10) : 0o12
hex(10) : 0xa
retour  : 10 10


Les préfixes `0b`, `0o`, `0x` indiquent la base. C'est aussi la raison de
l'erreur `leading zeros in decimal integer literals` rencontrée au notebook 04 :
Python réserve les zéros en tête à la notation octale d'autrefois.

## 4. Convertir une structure

`list`, `tuple`, `set` et `dict` convertissent d'une structure à l'autre.

In [133]:
liste = [1, 2, 3, 4, 5]

mon_tuple = tuple(liste)
print(mon_tuple, type(mon_tuple))

retour = list(mon_tuple)
print(retour, type(retour))

(1, 2, 3, 4, 5) <class 'tuple'>
[1, 2, 3, 4, 5] <class 'list'>


Attention au nom de variable. Écrire `liste_1 = tuple(quelque_chose)` produit
un tuple, quel que soit le nom donné. Le nom ne détermine pas le type, et une
variable mal nommée conduit à chercher un bug là où il n'est pas.

Vérifie toujours avec `type()` plutôt que de se fier au nom.

### set : supprimer les doublons

In [134]:
avec_doublons = [1, 2, 2, 3, 3, 3, 4]

print('set :', set(avec_doublons))
print('liste :', list(set(avec_doublons)))

set : {1, 2, 3, 4}
liste : [1, 2, 3, 4]


Rappel du notebook 06 : un ensemble perd l'ordre, et cet ordre change d'une
session à l'autre. Pour dédoublonner en conservant l'ordre :

```python
list(dict.fromkeys(avec_doublons))
```

### Convertir un dictionnaire

In [135]:
inventaire = {'bananes': 1000, 'mangues': 500, 'cerises': 800}

print(list(inventaire)) # les clés
print(list(inventaire.keys()))
print(list(inventaire.values()))
print(list(inventaire.items()))

['bananes', 'mangues', 'cerises']
['bananes', 'mangues', 'cerises']
[1000, 500, 800]
[('bananes', 1000), ('mangues', 500), ('cerises', 800)]


`list(dictionnaire)` donne les clés c'est cohérent avec `for cle in
dictionnaire` du notebook 05.

Et dans l'autre sens, `dict` reconstruit depuis des couples :

In [136]:
couples = [('a', 1), ('b', 2)]
print(dict(couples))

print(dict(zip(['x', 'y'], [10, 20])))

{'a': 1, 'b': 2}
{'x': 10, 'y': 20}


## 5. input et le formatage de texte

`input` lit une saisie au clavier. Elle renvoie toujours une chaîne, même si
l'utilisateur tape un nombre.

```python
x = input('Entrez un nombre : ')
print(type(x)) # <class 'str'>
```

D'où la nécessité de convertir :

```python
x = int(input('Entrez un nombre : '))
```

Deux mises en garde. Si l'utilisateur ne tape pas un nombre, `int` lève une
`ValueError` un `try` est indispensable dans un vrai programme.

Et dans un notebook publié, `input` bloque l'exécution en attendant une saisie.
C'est incompatible avec une réexécution automatique. Pour un notebook GitHub,
mieux vaut fixer les valeurs en dur.


In [137]:
x = input('Entrez un nombre : ')
print(type(x)) # <class 'str'>

Entrez un nombre : 7
<class 'str'>


In [138]:
x = int(input('Entrez un nombre : '))
print(type(x))

Entrez un nombre : 8
<class 'int'>


### Trois façons d'insérer des valeurs dans du texte

In [139]:
x = 30
ville = 'Paris'

print('La température est de ' + str(x) + ' degrés à ' + ville)
print('La température est de {} degrés à {}'.format(x, ville))
print(f'La température est de {x} degrés à {ville}')

La température est de 30 degrés à Paris
La température est de 30 degrés à Paris
La température est de 30 degrés à Paris


| Méthode | Remarque |
|---|---|
| concaténation `+` | exige `str()` sur les nombres, illisible |
| `.format()` | ancienne méthode, encore très répandue |
| f-string | depuis Python 3.6, la plus lisible |

La f-string est à préférer partout. Elle accepte des expressions et les formats
vus au notebook 02 :

In [140]:
temperature = 23.456
print(f'{temperature:.1f} °C')
print(f'{temperature:>10.2f} |')
print(f'Le double : {temperature * 2:.1f}')
print(f'{0.4237:.1%}')

23.5 °C
     23.46 |
Le double : 46.9
42.4%


Une astuce de débogage, depuis Python 3.8 : le `=` affiche le nom et la
valeur.

In [141]:
resultat = 42
print(f'{resultat=}')
print(f'{temperature * 2 = :.2f}')

resultat=42
temperature * 2 = 46.91


## 6. Construire des noms de clés dynamiquement

Un cas concret : les paramètres d'un réseau de neurones, stockés sous les clés
`W1`, `b1`, `W2`, `b2`.

In [142]:
import numpy as np

parametres = {
    'W1': np.random.randn(2, 4),
    'b1': np.random.randn(2, 1),
    'W2': np.random.randn(2, 2),
    'b2': np.random.randn(2, 1),
}

assert len(parametres) == 4, 'Une clé a été écrasée'
print('Clés :', list(parametres.keys()))

Clés : ['W1', 'b1', 'W2', 'b2']


L'`assert` reprend la leçon du notebook 05 : une clé écrite deux fois écrase la
première sans message. Ici, écrire `b1` au lieu de `b2` donnerait trois clés.

Pour parcourir les couches, on construit le nom de la clé à partir du numéro.

In [143]:
for i in range(1, 3):
    print(f' Couche {i} ')
    print('W de forme', parametres[f'W{i}'].shape)
    print('b de forme', parametres[f'b{i}'].shape)

 Couche 1 
W de forme (2, 4)
b de forme (2, 1)
 Couche 2 
W de forme (2, 2)
b de forme (2, 1)


Trois écritures équivalentes :

```python
parametres[f'W{i}'] # f-string, la plus lisible
parametres['W{}'.format(i)] # format
parametres['W' + str(i)] # concaténation, exige le str()
```

La dernière rappelle qu'on ne peut pas concaténer une chaîne et un entier :

```python
'W' + 1 # TypeError: can only concatenate str (not "int") to str
```

Cette technique est très utilisée dans le code de réseaux de neurones écrits à
la main, où le nombre de couches est variable.

## 7. Lire et écrire des fichiers

`open` ouvre un fichier. Le second argument précise le mode.

| Mode | Effet |
|---|---|
| `'r'` | lecture, erreur si le fichier n'existe pas |
| `'w'` | écriture, écrase le contenu existant |
| `'a'` | ajout à la fin |
| `'x'` | création, erreur si le fichier existe déjà |

Le mode `'w'` est destructeur : ouvrir un fichier existant en écriture le vide
immédiatement, avant même d'écrire quoi que ce soit.

### La forme à éviter

In [144]:
f = open('fichier.txt', 'w')
f.write('Bonjour')
f.close() # obligatoire, et facile à oublier

f = open('fichier.txt', 'r')
print(f.read())
f.close()

Bonjour


Si une erreur survient entre l'ouverture et le `close`, le fichier reste ouvert.
Sur un programme qui tourne longtemps, ça finit par saturer les descripteurs du
système.

### La forme correcte : with

`with` ferme le fichier automatiquement, même en cas d'erreur.

In [145]:
with open('fichier.txt', 'w') as f:
    f.write('Bonjour tout le monde')

with open('fichier.txt', 'r') as f:
    print(f.read())

Bonjour tout le monde


C'est la forme à utiliser systématiquement. À la sortie du bloc indenté, le
fichier est fermé quoi qu'il arrive.

### Trois erreurs classiques

In [146]:
# 1. write n'accepte qu'une chaîne
with open('fichier.txt', 'w') as f:
    try:
        f.write(1, 2)
    except TypeError as e:
        print('TypeError :', e)

TypeError : TextIOWrapper.write() takes exactly one argument (2 given)


In [147]:
# 2. write n'accepte pas les nombres
with open('fichier.txt', 'w') as f:
    try:
        f.write(42)
    except TypeError as e:
        print('TypeError :', e)

TypeError : write() argument must be str, not int


In [148]:
# 3. On ne peut pas lire un fichier ouvert en écriture
with open('fichier.txt', 'w') as f:
    f.write('test')
    try:
        f.read()
    except Exception as e:
        print(type(e).__name__, ':', e)

UnsupportedOperation : not readable


La troisième est la plus courante : on ouvre en `'w'`, on écrit, et on veut
vérifier en lisant dans la foulée. Il faut refermer et rouvrir en `'r'`.

### Écrire plusieurs lignes

`write` n'ajoute pas de saut de ligne. Sans `\n` explicite, tout se retrouve sur
une seule ligne.

In [149]:
with open('fichier.txt', 'w') as f:
    for i in range(5):
        f.write(f'{i}^2 = {i**2}') # sans \n

with open('fichier.txt', 'r') as f:
    print(f.read())

0^2 = 01^2 = 12^2 = 43^2 = 94^2 = 16


In [150]:
with open('fichier.txt', 'w') as f:
    for i in range(5):
        f.write(f'{i}^2 = {i**2}\n') # avec \n

with open('fichier.txt', 'r') as f:
    print(f.read())

0^2 = 0
1^2 = 1
2^2 = 4
3^2 = 9
4^2 = 16



### Lire ligne par ligne

`read()` charge tout le fichier en mémoire. Sur un gros fichier, c'est
problématique.

In [151]:
with open('fichier.txt', 'r') as f:
    lignes = f.readlines()
print('readlines :', lignes)

with open('fichier.txt', 'r') as f:
    for ligne in f: # ne charge qu'une ligne à la fois
        print(ligne.strip())

readlines : ['0^2 = 0\n', '1^2 = 1\n', '2^2 = 4\n', '3^2 = 9\n', '4^2 = 16\n']
0^2 = 0
1^2 = 1
2^2 = 4
3^2 = 9
4^2 = 16


Parcourir directement le fichier est la meilleure méthode : elle fonctionne quelle
que soit la taille, même sur plusieurs gigaoctets.

`strip()` retire le `\n` de fin, sinon `print` en ajoute un second et tout est
espacé.

Note que `readlines` conserve les `\n` c'est visible dans l'affichage
ci-dessus.

### Un mot sur l'encodage

Sur du texte avec des accents, précise l'encodage pour éviter les surprises
selon le système :

```python
with open('fichier.txt', 'w', encoding='utf-8') as f:
```

En pratique, pour des données tabulaires, on utilise `pd.read_csv` plutôt
qu'`open` mais le mécanisme sous-jacent est celui-ci.

## 8. map, filter, sorted

Trois fonctions qui appliquent un traitement à toute une séquence.

### sorted

Vu au notebook 04 : il renvoie une nouvelle liste triée, sans toucher à
l'original.

In [152]:
notes = [12, 8, 15, 6, 18]

print(sorted(notes))
print(sorted(notes, reverse=True))
print('original intact :', notes)

[6, 8, 12, 15, 18]
[18, 15, 12, 8, 6]
original intact : [12, 8, 15, 6, 18]


L'argument `key` indique sur quoi trier, via une fonction appliquée à chaque
élément.

In [153]:
mots = ['banane', 'kiwi', 'pomme', 'fraise']

print(sorted(mots)) # alphabétique
print(sorted(mots, key=len)) # par longueur
print(sorted(mots, key=lambda m: m[-1])) # par dernière lettre

['banane', 'fraise', 'kiwi', 'pomme']
['kiwi', 'pomme', 'banane', 'fraise']
['banane', 'pomme', 'fraise', 'kiwi']


C'est l'usage typique des lambdas du notebook 02, et c'est ce qui servait à
trier un dictionnaire par valeur au notebook 05.

### map : appliquer une fonction à chaque élément

In [154]:
nombres = [1, 2, 3, 4]

doubles = map(lambda n: n * 2, nombres)
print(doubles)
print(list(doubles))

[2, 4, 6, 8]


`map` ne renvoie pas une liste mais un objet paresseux, comme le générateur du
notebook 06. Il faut `list()` pour voir le contenu.

Et il s'épuise de la même façon :

In [155]:
doubles = map(lambda n: n * 2, nombres)

print('premier appel :', list(doubles))
print('second appel  :', list(doubles))

premier appel : [2, 4, 6, 8]
second appel  : []


Liste vide au second appel, sans erreur.

### filter : ne garder que certains éléments

In [156]:
pairs = filter(lambda n: n % 2 == 0, range(10))
print(list(pairs))

[0, 2, 4, 6, 8]


### map/filter ou compréhension ?

Les deux font la même chose :

```python
list(map(lambda n: n*2, nombres)) # map
[n * 2 for n in nombres] # compréhension

list(filter(lambda n: n > 2, nombres)) # filter
[n for n in nombres if n > 2] # compréhension
```

Le guide de style Python recommande la compréhension. Elle est plus lisible,
surtout quand il faut combiner transformation et filtre :

```python
[n * 2 for n in nombres if n > 2] # clair
list(map(lambda n: n*2, filter(lambda n: n>2, nombres))) # imbriqué
```

`map` garde un intérêt avec une fonction déjà nommée, sans lambda :

```python
list(map(str, nombres)) # plus court que [str(n) for n in nombres]
list(map(int, saisies))
```

Tu croiseras `map` et `filter` dans du code existant, il faut savoir les lire.
Pour écrire, préfère les compréhensions.

## 9. Mémo

### Les fonctions du notebook

| Catégorie | Fonctions |
|---|---|
| Nombres | `abs`, `round`, `pow`, `min`, `max`, `sum` |
| Séquences | `len`, `sorted`, `reversed`, `enumerate`, `zip` |
| Booléens | `all`, `any`, `bool` |
| Types | `int`, `float`, `str`, `bool`, `type`, `isinstance` |
| Structures | `list`, `tuple`, `set`, `dict` |
| Bases | `bin`, `oct`, `hex` |
| Fonctionnel | `map`, `filter` |
| Entrées-sorties | `print`, `input`, `open` |

### Ce qui est faux en Python

`0`, `0.0`, `''`, `[]`, `{}`, `()`, `set()`, `None`, `False`.

Tout le reste est vrai, y compris `-19`, `'0'` et `[0]`.

### Les pièges

| Situation | Ce qui se passe |
|---|---|
| `round(2.5)` | donne 2, arrondi au pair |
| `round(2.675, 2)` | donne 2.67, à cause du binaire |
| `float(x)` sans réassigner | `x` reste inchangé |
| `all([0, 23])` | `False`, car 0 est faux |
| `all([])` | `True` |
| `open(f, 'w')` | vide le fichier immédiatement |
| `f.write(42)` | `TypeError`, il faut une chaîne |
| `write` sans `\n` | tout sur une seule ligne |
| `map` parcouru deux fois | vide au second passage |
| `input()` | renvoie toujours une chaîne |

## 10. Exercices

**Exercice 1**

Écris une fonction `statistiques(fichier)` qui lit un fichier de nombres, un par
ligne, et renvoie leur nombre, leur somme, leur moyenne, leur minimum et leur
maximum. Crée d'abord le fichier avec 100 nombres aléatoires.

Gère le cas des lignes vides et des lignes non numériques sans planter.

**Exercice 2**

Compare trois façons de doubler chaque élément d'une liste d'un million
d'entiers : une boucle avec `append`, une compréhension, et `map`. Mesure les
temps. Puis ajoute NumPy à la comparaison.

Lequel est le plus rapide, lequel est le plus lisible, et que conclus-tu ?

**Exercice 3**

Cette fonction contient deux pièges de ce notebook :

```python
def moyenne_positive(valeurs):
    if all(valeurs):
        return round(sum(valeurs) / len(valeurs))
    return None
```

Teste-la avec `[1, 2, 3]`, `[0, 1, 2]`, `[2.5, 2.5]` et `[]`. Explique chaque
résultat surprenant, puis corrige la fonction.

## Pour continuer

Le notebook suivant porte sur les modules de base : `math`, `random`,
`statistics`, `os`, `glob`. Ce sont des fonctions supplémentaires, disponibles
après un `import`.

Puis viendra la programmation orientée objet, et enfin NumPy où l'on retrouvera
`sum`, `min`, `max` en version vectorisée, des dizaines de fois plus rapides.

In [157]:
# Exercice 1

In [158]:
import random

random.seed(0) # pour un résultat reproductible

with open('nombres.txt', 'w') as f:
    for _ in range(100):
        f.write(f'{random.uniform(-50, 50):.2f}\n')
    f.write('\n') # ligne vide
    f.write('abc\n') # ligne non numérique
    f.write('\n') # espaces seulement

In [159]:
def statistiques(fichier):
    """Lit un fichier de nombres, un par ligne, et calcule ses statistiques.
    Les lignes vides sont ignorées, les lignes non numériques sont comptées
    à part. Retourne None si aucun nombre n'a pu être lu.
    """
    valeurs = []
    ignorees = 0

    with open(fichier, 'r') as f:
        for ligne in f: # ligne par ligne, pas read()
            ligne = ligne.strip()

            if not ligne: # chaîne vide = fausse
                continue

            try:
                valeurs.append(float(ligne))
            except ValueError:
                ignorees += 1

    if not valeurs: # liste vide = fausse
        return None

    return {
        'nombre': len(valeurs),
        'somme': sum(valeurs),
        'moyenne': sum(valeurs) / len(valeurs),
        'minimum': min(valeurs),
        'maximum': max(valeurs),
        'ignorees': ignorees,
    }

In [160]:
resultat = statistiques('nombres.txt')

for cle, valeur in resultat.items():
    print(f'{cle:<10} {valeur:>10.2f}' if isinstance(valeur, float)
          else f'{cle:<10} {valeur:>10}')

nombre            100
somme          856.72
moyenne          8.57
minimum        -49.89
maximum         49.63
ignorees            1


In [161]:
with open('nombres.txt') as f:
    valeurs = [float(l) for l in f if l.strip().replace('.','').replace('-','').isdigit()]

In [162]:
import statistics

print(statistics.mean(valeurs))
print(statistics.median(valeurs))
print(statistics.stdev(valeurs))

8.5672
8.870000000000001
26.582428618783634


In [163]:
# Exericice 2

In [164]:
import timeit

setup = 'd = list(range(1_000_000))'

boucle = timeit.timeit('r=[]\nfor x in d: r.append(x*2)', setup, number=3) / 3
comp = timeit.timeit('[x*2 for x in d]', setup, number=3) / 3
mp = timeit.timeit('list(map(lambda x: x*2, d))', setup, number=3) / 3
np_  = timeit.timeit('a*2', 'import numpy as np; a=np.arange(1_000_000)', number=3) / 3

print(f'boucle : {boucle:.4f} s')
print(f'compréhension : {comp:.4f} s')
print(f'map + lambda : {mp:.4f} s')
print(f'NumPy : {np_:.5f} s')
print(f'NumPy est {comp/np_:.0f}x plus rapide que la compréhension')

boucle : 0.0655 s
compréhension : 0.0713 s
map + lambda : 0.1134 s
NumPy : 0.00244 s
NumPy est 29x plus rapide que la compréhension


In [165]:
timeit.timeit('list(map(str, d))', setup, number=3) # rapide
timeit.timeit('[str(x) for x in d]', setup, number=3) # un peu plus lent

0.37920905300052254

In [166]:
# Exerice 3

In [167]:
def moyenne_positive(valeurs):
    if all(valeurs):
        return round(sum(valeurs) / len(valeurs))
    return None

In [168]:
for test in [[1, 2, 3], [0, 1, 2], [2.5, 2.5], []]:
    try:
        print(f'{str(test):<12} -> {moyenne_positive(test)}')
    except Exception as e:
        print(f'{str(test):<12} -> {type(e).__name__} : {e}')

[1, 2, 3]    -> 2
[0, 1, 2]    -> None
[2.5, 2.5]   -> 2
[]           -> ZeroDivisionError : division by zero


In [169]:
def moyenne_positive(valeurs, decimales=2):
    """Moyenne d'une liste, si toutes les valeurs sont strictement positives.
    Retourne None si la liste est vide ou contient une valeur négative ou nulle.
    """
    if not valeurs: # liste vide : cas explicite
        return None

    if not all(v > 0 for v in valeurs): # positivité, pas "vérité"
        return None

    return round(sum(valeurs) / len(valeurs), decimales)

In [170]:
for test in [[1, 2, 3], [0, 1, 2], [2.5, 2.5], [], [-5, 3], [1, 2, 4]]:
    print(f'{str(test):<12} -> {moyenne_positive(test)}')

[1, 2, 3]    -> 2.0
[0, 1, 2]    -> None
[2.5, 2.5]   -> 2.5
[]           -> None
[-5, 3]      -> None
[1, 2, 4]    -> 2.33


In [171]:
all(v > 0 for v in valeurs) # explicite
any(v is None for v in valeurs) # explicite
all(valeurs) # ambigu, à éviter

True